# 08 · Các cú nhảy vị trí còn sót lại

**Câu hỏi:** sau khi làm sạch, vẫn còn những bước nhảy > 1100 km/h giữa hai điểm liên tiếp. Bước loại điểm gai chỉ bắt được **một điểm** nhảy đi rồi về. Bước cắt segment hiện chỉ **cắt rời** quỹ đạo tại đó và giữ nguyên các điểm. Có cần một quy tắc làm sạch mới không, và nó có làm sai stay-point không?

**Hai loại:**
- **Khối lệch** (`excursion`): quỹ đạo nhảy đi ở điểm k, nhảy về ở điểm m, và hai điểm ngoài (k và m+1) nối với nhau hợp lý (≤ 1100 km/h). Giống điểm gai nhưng dài nhiều điểm.
- **Lệch một phía** (`one_sided`): nhảy đi và không quay về trong file.

**Phía nào sai?** Vật lý không trả lời được: chỉ biết hai phía không thể cùng đúng với mốc thời gian đã ghi. Dùng thêm **dữ liệu thật** của chính user:
- **Chỗ quen:** phía đó có nằm trong 200 m quanh một điểm của user ở **file khác** không. Lỗi GPS thường rơi vào chỗ lạ.
- **Phương tiện và thời gian** của bước nhảy.
- **Tác động:** bao nhiêu stay-point (cách chạy của `src/`: `detect()`, 200 m / 30 phút, cắt tại khoảng ngắt > 18 h) chạm vào phần bị dời.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK = "08_trajectory_jumps"
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("outputs") / NOTEBOOK
FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR = (OUT_DIR / d for d in ("figures", "tables", "cache", "maps"))
for d in (FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(Path("../src").resolve()))
import folium

from gps.data.processing import DEFAULT_THRESHOLDS, _haversine_vectorized
from gps.features.stay_point import MAX_GAP_HOURS, DistanceMode, StayPointDetector

USERS_DIR = Path("../data/processed/users")
V_MAX = DEFAULT_THRESHOLDS.impossible_speed_kmh
FAMILIAR_M = 200        # stay-point radius


def load_labels(uid: str) -> pd.DataFrame | None:
    path = DATA_DIR / uid / "labels.txt"
    if not path.exists():
        return None
    lab = pd.read_csv(path, sep="\t", skiprows=1, header=None, names=["start", "end", "mode"])
    lab["start"] = pd.to_datetime(lab["start"], format="%Y/%m/%d %H:%M:%S", errors="coerce")
    lab["end"] = pd.to_datetime(lab["end"], format="%Y/%m/%d %H:%M:%S", errors="coerce")
    return lab.dropna()

## 1. Tìm và phân loại các cú nhảy
Với mỗi bước nhảy đi (k → k+1), tìm bước nhảy về gần nhất (m → m+1) sao cho k và m+1 nối được với nhau ≤ 1100 km/h. Tìm được thì đó là khối lệch; không tìm được thì là lệch một phía (phần bị dời tính tới cuối file).

In [ ]:
JUMPS_PATH = CACHE_DIR / "jumps.parquet"
if JUMPS_PATH.exists():
    jumps = pd.read_parquet(JUMPS_PATH)
else:
    rows = []
    for path in sorted(USERS_DIR.glob("user_*.parquet")):
        uid = path.stem.split("_")[1]
        df = pd.read_parquet(path, columns=["source_file", "datetime", "lat", "lon"])
        for fname, f in df.groupby("source_file"):
            if len(f) < 2:
                continue
            f = f.sort_values("datetime")
            t = f["datetime"].to_numpy().astype("datetime64[ns]").astype(np.int64) / 1e9
            la, lo, times = f["lat"].to_numpy(), f["lon"].to_numpy(), f["datetime"].to_numpy()
            step = _haversine_vectorized(la[:-1], lo[:-1], la[1:], lo[1:])
            out_idx = list(np.flatnonzero(step / np.diff(t) * 3.6 > V_MAX))
            used = set()
            for n, k in enumerate(out_idx):
                if k in used:
                    continue
                back = next((m for m in out_idx[n + 1:] if m not in used
                             and _haversine_vectorized(la[k], lo[k], la[m + 1], lo[m + 1]) / max(t[m + 1] - t[k], 1e-9) * 3.6 <= V_MAX), None)
                end = back if back is not None else len(f) - 1
                used |= {k} | ({back} if back is not None else set())
                rows.append({"user_id": uid, "file": fname, "kind": "excursion" if back is not None else "one_sided",
                             "t_start": times[k + 1], "t_end": times[end], "n_points": end - k,
                             "duration_min": (t[end] - t[k + 1]) / 60, "jump_km": step[k] / 1000, "jump_dt_s": t[k + 1] - t[k],
                             "moved_lat": np.median(la[k + 1:end + 1]), "moved_lon": np.median(lo[k + 1:end + 1]),
                             "other_lat": np.median(np.r_[la[:k + 1], la[end + 1:]]), "other_lon": np.median(np.r_[lo[:k + 1], lo[end + 1:]]),
                             "file_points": len(f)})
    jumps = pd.DataFrame(rows)
    jumps.to_parquet(JUMPS_PATH)

overview = jumps.groupby("kind").agg(cases=("user_id", "size"), users=("user_id", "nunique"),
                                     median_points=("n_points", "median"), max_points=("n_points", "max"),
                                     median_minutes=("duration_min", "median"), max_minutes=("duration_min", "max"),
                                     median_jump_km=("jump_km", "median"), max_jump_km=("jump_km", "max"),
                                     median_jump_dt_s=("jump_dt_s", "median"))
overview.to_csv(TABLE_DIR / "jump_overview.csv")
print("Cases by user (top 8):", jumps.groupby("user_id").size().sort_values(ascending=False).head(8).to_dict())
overview.round(2)

## 2. Phương tiện lúc nhảy
Nhãn của chuyến đi chứa thời điểm nhảy (nếu user có `labels.txt`).

In [ ]:
modes = []
for r in jumps.itertuples():
    lab = load_labels(r.user_id)
    if lab is None:
        modes.append("no_labels")
        continue
    hit = lab.loc[(lab["start"] <= r.t_start) & (lab["end"] >= r.t_start), "mode"]
    modes.append(hit.iloc[0] if len(hit) else "unlabelled_time")
jumps["mode_at_jump"] = modes
mode_table = pd.crosstab(jumps["mode_at_jump"], jumps["kind"])
mode_table.to_csv(TABLE_DIR / "jump_mode.csv")
mode_table

## 3. Phía nào nằm ở chỗ quen?
`moved_familiar`: phần bị dời nằm trong 200 m quanh một điểm của user ở file khác. `other_familiar`: phần còn lại của file (trước và sau phần bị dời) cũng được kiểm tra như vậy. Nếu một phía là lỗi GPS, phía đó phải ít "quen" hơn hẳn phía kia.

In [ ]:
def nearest_other_m(uid: str, fname: str, lat: float, lon: float, cache: dict = {}) -> float:
    if uid not in cache:
        cache.clear()
        cache[uid] = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["source_file", "lat", "lon"])
    o = cache[uid][cache[uid]["source_file"] != fname]
    if o.empty or np.isnan(lat):
        return np.nan
    o = o.iloc[:: max(1, len(o) // 50000)]
    return float(_haversine_vectorized(np.full(len(o), lat), np.full(len(o), lon), o["lat"].to_numpy(), o["lon"].to_numpy()).min())


jumps = jumps.sort_values("user_id", kind="stable")
jumps["moved_familiar"] = [nearest_other_m(r.user_id, r.file, r.moved_lat, r.moved_lon) <= FAMILIAR_M for r in jumps.itertuples()]
jumps["other_familiar"] = [nearest_other_m(r.user_id, r.file, r.other_lat, r.other_lon) <= FAMILIAR_M for r in jumps.itertuples()]
familiar = jumps.groupby("kind")[["moved_familiar", "other_familiar"]].mean()
familiar.to_csv(TABLE_DIR / "jump_side_familiarity.csv")
print(pd.crosstab([jumps["kind"], jumps["other_familiar"]], jumps["moved_familiar"]).to_string())
familiar.round(3)

## 4. Tác động lên stay-point
Chạy `detect()` (200 m / 30 phút, cắt tại khoảng ngắt > 18 h) trên các chuỗi file có chứa cú nhảy, rồi đếm stay-point có thời gian chạm vào phần bị dời. Một stay-point không thể vắt qua cú nhảy, vì hai phía cách nhau hơn 300 m.

In [ ]:
STAYS_PATH = CACHE_DIR / "stays_affected_chains.parquet"
if STAYS_PATH.exists():
    stays = pd.read_parquet(STAYS_PATH)
else:
    det = StayPointDetector(30, 200, DistanceMode.CENTROID)
    rows = []
    for uid, uj in jumps.groupby("user_id"):
        df = pd.read_parquet(USERS_DIR / f"user_{uid}.parquet", columns=["source_file", "datetime", "lat", "lon"])
        b = df.groupby("source_file")["datetime"].agg(["min", "max"]).sort_values("min")
        b["chain"] = ((b["min"] - b["max"].shift()).dt.total_seconds() / 3600 > MAX_GAP_HOURS).cumsum()
        files = b.index[b["chain"].isin(b.loc[uj["file"].unique(), "chain"])]
        for sp in det.detect(df[df["source_file"].isin(files)]):
            rows.append({"user_id": uid, "arrival": pd.Timestamp(sp.arrival_time), "departure": pd.Timestamp(sp.departure_time),
                         "duration_min": sp.duration_minutes})
    stays = pd.DataFrame(rows)
    stays.to_parquet(STAYS_PATH)

touched = []
for r in jumps.itertuples():
    s = stays[stays["user_id"] == r.user_id]
    hit = (s["arrival"] <= r.t_end) & (s["departure"] >= r.t_start)
    touched.append((int(hit.sum()), float(s.loc[hit, "duration_min"].sum() / 60)))
jumps["stays_touched"], jumps["stay_hours_touched"] = zip(*touched)
impact = jumps.groupby("kind")[["stays_touched", "stay_hours_touched"]].sum()
impact["share_of_stay_hours_in_affected_chains"] = impact["stay_hours_touched"] / (stays["duration_min"].sum() / 60)
impact.to_csv(TABLE_DIR / "jump_staypoint_impact.csv")
print(f"Stay-points in chains that contain a jump: {len(stays):,} ({stays['duration_min'].sum() / 60:,.0f} h)")
impact.round(4)

## 5. Bản đồ ví dụ
Một khối lệch và một lần lệch một phía: chấm xanh là phần còn lại của file, chấm đỏ là phần bị dời, đường đứt nối hai điểm ở hai đầu cú nhảy. Mở file `.html` trong `outputs/08_trajectory_jumps/maps/`.

In [ ]:
examples = pd.concat([jumps[jumps["kind"] == "excursion"].sort_values("n_points").iloc[[len(jumps[jumps["kind"] == "excursion"]) // 2]],
                      jumps[jumps["kind"] == "one_sided"].sort_values("n_points").iloc[[len(jumps[jumps["kind"] == "one_sided"]) // 2]]])
for r in examples.itertuples():
    f = pd.read_parquet(USERS_DIR / f"user_{r.user_id}.parquet", columns=["source_file", "datetime", "lat", "lon"])
    f = f[f["source_file"] == r.file].sort_values("datetime")
    moved = (f["datetime"] >= r.t_start) & (f["datetime"] <= r.t_end)
    m = folium.Map(location=[f["lat"].median(), f["lon"].median()], zoom_start=12)
    for part, color in ((f[~moved], "#1f77b4"), (f[moved], "#d62728")):
        for p in part.iloc[:: max(1, len(part) // 300)].itertuples():
            folium.CircleMarker([p.lat, p.lon], radius=2, color=color, fill=True,
                                tooltip=f"{p.datetime:%Y-%m-%d %H:%M:%S} GMT").add_to(m)
    i = int(np.flatnonzero(moved.to_numpy())[0])
    folium.PolyLine(f[["lat", "lon"]].iloc[[i - 1, i]].to_numpy(), color="black", dash_array="6", weight=2,
                    tooltip=f"jump {r.jump_km:.1f} km in {r.jump_dt_s:.0f} s").add_to(m)
    out = MAP_DIR / f"{r.kind}_user{r.user_id}_{Path(r.file).stem}.html"
    m.save(out)
    print(f"{r.kind}: user {r.user_id}, {r.n_points} moved points, jump {r.jump_km:.1f} km in {r.jump_dt_s:.0f} s -> {out}")

## Kết luận

| | Khối lệch | Lệch một phía |
|---|---|---|
| Số trường hợp | 58 (17 user) | 250 (74 user; riêng user 083 có 46) |
| Phần bị dời (trung vị) | 64 điểm, 8 phút; lớn nhất 19,198 điểm, 15 giờ | 572 điểm, 70 phút |
| Bước nhảy (trung vị) | 3.7 km trong 5 giây | 1.8 km trong 4 giây |
| Phần bị dời ở chỗ quen / phía còn lại ở chỗ quen | 26% / 26% | 62% / 78% |
| Stay-point chạm vào | **11** (20 giờ, tức 0.03% tổng thời gian lưu trú toàn bộ dữ liệu) | **300** (1,105 giờ, tức 1.4% toàn bộ dữ liệu) |

**Khối lệch:** phép thử "chỗ quen" **không phân biệt được** phía nào sai (cả hai phía đều 26%), nên không có bằng chứng để chọn xoá phía nào. Một quy tắc xoá khối lệch sẽ cần thêm một ngưỡng độ dài khối không có bằng chứng, và có thể xoá nhầm cả khối 15 giờ là dữ liệu thật. Đổi lại, tác động lên stay-point **không đáng kể**: 11 stay-point, 0.03% thời gian lưu trú.

**Lệch một phía:** bước nhảy rất ngắn về thời gian (trung vị 4 giây) và cả hai phía phần lớn nằm ở chỗ quen. Nhiều khả năng **toạ độ đúng** và chỉ mốc thời gian ngay lúc thiết bị bắt lại tín hiệu bị sai (ra khỏi hầm, toà nhà, tàu điện ngầm). Một stay-point không thể vắt qua cú nhảy (hai phía cách nhau hơn 300 m, bán kính chỉ 200 m), nên stay-point ở mỗi phía vẫn là một lần ở lại hợp lệ.

**Quyết định: không thêm quy tắc làm sạch.** Giữ cách xử lý hiện tại: điểm được giữ, quỹ đạo bị cắt segment tại cú nhảy (`segment_trajectories`). Bằng chứng không cho biết phía nào sai, và tác động lên stay-point đủ nhỏ để không cần đoán.

**Ghi chú:**
- 3 nhóm trùng còn mang lý do `duplicate_invalid_anchors` (notebook 02, mục 3.3) chính là các cú nhảy kéo dài ở đây; chúng tiếp tục nằm trong quarantine.
- User 083 chiếm 46/250 lần lệch một phía, có thể do thiết bị riêng. Nên xem lại khi đánh giá classifier theo từng user.